In [1]:
import os
from pathlib import Path
os.chdir(Path.cwd().parent)
print(f"Current directory: {Path.cwd()}")

Current directory: /home/asaldivargarci1064/projects/h_lacustris


In [2]:
import warnings
# Suppress all warnings
warnings.filterwarnings("ignore")

import numpy as np
import polars as pl

from lmfit import Model
from tqdm import tqdm

In [6]:
def load_biolog_data(
    files: list[str], plate_idx, repl_str, repl_end
) -> pl.DataFrame:
    plate_key = {"1": "PM1", "2": "PM2", "N": "PM3"}
    n_wells = 96
    col_letters = "A:AF"
    
    # Get all the data
    start_row = 30
    read_options = {
        "header_row": start_row,
        "n_rows": n_wells,
        "use_columns": col_letters,
    }
    df_dict = pl.read_excel(
        file,
        sheet_id=0,
        read_options=read_options,
        has_header=True,
        infer_schema_length=5,
    )
    for k, v in df_dict.items():
        df_dict[k] = v.rename({"Wavel.": "well"})
    
    # Get all dates
    date_col = "B"
    start_row = 26
    read_options = {
        "skip_rows": start_row,
        "n_rows": 1,
        "use_columns": date_col,
    }
    dates = pl.read_excel(file, sheet_id=0, read_options=read_options)
    
    # Annotate the data with replicate and plate info
    for sheet, v in df_dict.items():
        try:
            plate = plate_key[sheet[plate_idx]]
        except KeyError:
            print(sheet)
            raise KeyError
        replicate = sheet[repl_str:repl_end]
        dates[sheet] = dates[sheet].with_columns(
                    pl.col("__UNNAMED__1").str.to_datetime("%m/%d/%Y %I:%M:%S %p"),
        )
        df_dict[sheet] = df_dict[sheet].with_columns(
            plate=pl.lit(plate),
            replicate=pl.lit(replicate),
            date=dates[sheet][0, 0]
        )
    
    od_df = pl.concat(df_dict.values()).sort("date")
    od_df = od_df.with_columns(
            time = pl.col("date") - od_df[0, -1],
        )
    
    return od_df.with_columns(time = pl.col("time").dt.total_seconds() / (3600 * 24))

def gompertz(t, A, B, C):
    """Get gompertz model."""
    return A * np.exp(-np.exp((B * np.e / A) * (C - t) + 1))

def fit_gompertz(dataframe):
    """Fit data to gompertz model.

    Parameters
    ----------
    t: time data
    y: growth data
    A, B, C: Initial conditions for gompertz Parameters

    """
    # Initial conditions
    A=1.0
    B=0.03
    C=3
    y1 = dataframe["680"].to_numpy()
    t1 = dataframe["time"].to_numpy()
    dataframe = dataframe.drop_nans(subset="logratio")
    y2 = dataframe["logratio"].to_numpy()
    t2 = dataframe["time"].to_numpy()
    model = Model(gompertz)
    params = model.make_params(A=A, B=B, C=C)
    params["A"].min = 0.001
    params["B"].min = 0
    params["C"].min = 0
    params["A"].max = 100
    params["B"].max = 2
    params["C"].max = 100
    #result1 = model.fit(y1, params, t=t1, method="differential_evolution")
    try:
        result2 = model.fit(y2, params, t=t2, method="differential_evolution")
        result_dict = {
            "model": "gompertz",
            "mu": result2.params["B"].value,
            "r2": result2.rsquared
        }
    except TypeError:
        result_dict = {
            "model": "gompertz",
            "mu": np.nan,
            "r2": np.nan
        }
    except AttributeError:
        result_dict = {
            "model": "gompertz",
            "mu": np.nan,
            "r2": np.nan
        }
    except RuntimeError:
        result_dict = {
            "model": "gompertz",
            "mu": np.nan,
            "r2": np.nan
        }
    
    return result_dict

def fit_log(dataframe):
    """Fit data to exponetial model.

    Parameters
    ----------
    t: time data
    y: growth data
    m: growth rate
    """
    def linear_model(t, m, b):
        return m * t + b
    # Initial guess
    m=0.03
    b=0.05
    # Log transform the data
    y = np.log(dataframe["680"].to_numpy())
    t = dataframe["time"].to_numpy()
    model = Model(linear_model)
    params = model.make_params(m=m, b=b)
    params["m"].min = 0
    result = model.fit(y, params, t=t)

    return {
        "model": "log",
        "mu": result.params["m"].value,
        "r2": result.rsquared
    }
    

### Load the data
____

In [8]:
# Light condition
file = "data/0_raw/tecan/E260123.xlsx"
data_df = load_biolog_data(file, -1, 3, 5)
print(data_df.describe())
print(data_df.head())

shape: (9, 37)
┌────────────┬──────┬──────────┬──────────┬───┬───────┬───────────┬─────────────────────┬──────────┐
│ statistic  ┆ well ┆ 400      ┆ 420      ┆ … ┆ plate ┆ replicate ┆ date                ┆ time     │
│ ---        ┆ ---  ┆ ---      ┆ ---      ┆   ┆ ---   ┆ ---       ┆ ---                 ┆ ---      │
│ str        ┆ str  ┆ f64      ┆ f64      ┆   ┆ str   ┆ str       ┆ str                 ┆ f64      │
╞════════════╪══════╪══════════╪══════════╪═══╪═══════╪═══════════╪═════════════════════╪══════════╡
│ count      ┆ 7776 ┆ 7776.0   ┆ 7776.0   ┆ … ┆ 7776  ┆ 7776      ┆ 7776                ┆ 7776.0   │
│ null_count ┆ 0    ┆ 0.0      ┆ 0.0      ┆ … ┆ 0     ┆ 0         ┆ 0                   ┆ 0.0      │
│ mean       ┆ null ┆ 0.433446 ┆ 0.40982  ┆ … ┆ null  ┆ null      ┆ 2026-01-27          ┆ 4.122522 │
│            ┆      ┆          ┆          ┆   ┆       ┆           ┆ 17:13:16.864197     ┆          │
│ std        ┆ null ┆ 0.216355 ┆ 0.203425 ┆ … ┆ null  ┆ null      ┆ null    

### Correct OD with pathlenght
---

In [10]:
# Volume for each well
data_df = data_df.with_columns(
    volume=(pl.col("980") - pl.col("900"))/0.0010018,
)
# Pathlenght for each well
data_df = data_df.with_columns(
    pathlenght=(pl.col("volume") * 0.0055) + 0.0128
)
# Corrected OD dataframe
wavelenghts = ["440", "680"]
cols = ["well", "plate", "replicate", "date", "time", "volume", "pathlenght"]
corrected_df = data_df.select(cols+wavelenghts).with_columns(
    pl.col("440")/pl.col("pathlenght"),
    pl.col("680")/pl.col("pathlenght"),
)
corrected_df.head()

well,plate,replicate,date,time,volume,pathlenght,440,680
str,str,str,datetime[μs],f64,f64,f64,f64,f64
"""A1""","""PM1""","""P1""",2026-01-23 14:16:51,0.0,98.023551,0.55193,0.284275,0.259997
"""A2""","""PM1""","""P1""",2026-01-23 14:16:51,0.0,98.123373,0.552479,0.360195,0.27259
"""A3""","""PM1""","""P1""",2026-01-23 14:16:51,0.0,99.321222,0.559067,0.296029,0.264548
"""A4""","""PM1""","""P1""",2026-01-23 14:16:51,0.0,99.2214,0.558518,0.291486,0.260332
"""A5""","""PM1""","""P1""",2026-01-23 14:16:51,0.0,99.2214,0.558518,0.286294,0.261943


### Fit to gompertz and linear models
---

In [11]:
cols = ["plate", "well", "replicate"]
gpb = corrected_df.group_by("plate", "well", "replicate")
data_to_concat = []
fits_to_concat = []
for name, data in tqdm(gpb):
    # Apply log(od_t / od_time_zero) to each well
    # Get log ratio
    data_tf = data.with_columns(
        logratio=np.log(pl.col("680")/pl.col("680").first())
    )
    # Replace negative values with nan
    data_tf = data_tf.with_columns(
        logratio=pl.when(pl.col("logratio") < 0).then(np.nan).otherwise(pl.col("logratio"))
    )
    # Save transformed data
    data_to_concat.append(data_tf)

    # Fit to the gompertz model
    keys = dict(zip(cols, name))
    fit_results = fit_gompertz(data_tf)
    keys.update(fit_results)
    fits_to_concat.append(pl.DataFrame(keys))
    # Fit to the linear model
    keys = dict(zip(cols, name))
    fit_results_l = fit_log(data_tf)
    keys.update(fit_results_l)
    fits_to_concat.append(pl.DataFrame(keys))
    
corrected_df = pl.concat(data_to_concat)
fits_df = pl.concat(fits_to_concat)
fits_df

864it [01:33,  9.23it/s]


plate,well,replicate,model,mu,r2
str,str,str,str,f64,f64
"""PM2""","""C3""","""P1""","""gompertz""",0.19575,0.384762
"""PM2""","""C3""","""P1""","""log""",0.141066,0.446225
"""PM2""","""A7""","""P1""","""gompertz""",0.161709,0.912091
"""PM2""","""A7""","""P1""","""log""",0.134029,0.931382
"""PM3""","""B12""","""P2""","""gompertz""",0.451173,0.944512
…,…,…,…,…,…
"""PM3""","""E3""","""P3""","""log""",0.016747,0.526131
"""PM3""","""H2""","""P1""","""gompertz""",0.265168,0.848756
"""PM3""","""H2""","""P1""","""log""",0.218738,0.910658


In [12]:
corrected_df

well,plate,replicate,date,time,volume,pathlenght,440,680,logratio
str,str,str,datetime[μs],f64,f64,f64,f64,f64,f64
"""C3""","""PM2""","""P1""",2026-01-23 14:51:23,0.023981,99.121578,0.557969,0.293565,0.262201,0.0
"""C3""","""PM2""","""P1""",2026-01-24 11:01:18,0.864201,90.33739,0.509656,0.789749,0.604722,0.835657
"""C3""","""PM2""","""P1""",2026-01-25 12:41:46,1.93397,69.474931,0.394912,1.450956,0.933372,1.269692
"""C3""","""PM2""","""P1""",2026-01-26 14:57:22,3.028137,52.804967,0.303227,2.748433,1.878788,1.969271
"""C3""","""PM2""","""P1""",2026-01-27 09:20:21,3.794097,51.008201,0.293345,2.686597,1.836404,1.946453
…,…,…,…,…,…,…,…,…,…
"""G5""","""PM2""","""P3""",2026-01-27 09:44:44,3.81103,89.538829,0.505264,0.545062,0.417604,0.504547
"""G5""","""PM2""","""P3""",2026-01-28 10:56:00,4.860521,87.742077,0.495381,0.55755,0.428761,0.530912
"""G5""","""PM2""","""P3""",2026-01-29 10:46:57,5.854236,85.845474,0.48495,0.565625,0.432828,0.540354


In [13]:
out_path = Path("data/1_interim/biolog/E260123b/")
out_path.mkdir(parents=True, exist_ok=True)
corrected_df.write_csv(out_path / "data_df.csv")
fits_df.write_csv(out_path / "rates_df.csv")
print("Data saved to: data/1_interim/")

Data saved to: data/1_interim/
